# Joe — Google Colab API

Este notebook transforma uma sessão do Google Colab em um backend remoto temporário para o aplicativo desktop Joe.

## O que ele inicia
- **Texto → Áudio:** Kyutai Pocket TTS (CPU).
- **Áudio → Texto:** Moondream Parakeet Redux via Photon (usa CUDA quando uma GPU está disponível).
- **API:** FastAPI protegida por token Bearer.
- **Acesso externo:** Cloudflare Quick Tunnel com URL temporária `trycloudflare.com`.

## Antes de executar
1. Em **Ambiente de execução → Alterar o tipo de ambiente de execução**, escolha GPU quando disponível (por exemplo, T4).
2. Execute as células em sequência.
3. Na última célula, copie a URL da API e o token para a aba **Google Colab / API** do Joe.
4. Não compartilhe a saída da célula com o token. Qualquer pessoa que tenha a URL e o token poderá consumir sua sessão.

**Importante:** o Colab não oferece disponibilidade ou duração garantidas. A sessão e o túnel podem ser encerrados; ao reconectar, gere uma URL e um token novos. <https://research.google.com/colaboratory/faq.html>


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/joe')
REPO_URL = 'https://github.com/g4b3r-desktop/joe.git'
BRANCH = 'feature/colab-model-api'

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    # Atualiza o clone quando o notebook é reutilizado na mesma sessão do Colab.
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

print('Código do Joe pronto:', REPO_DIR)


In [ ]:
%pip install -q -r /content/joe/colab/requirements.txt

import torch
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Aviso: sem GPU disponível; o Parakeet será executado na CPU.')


## Google Drive: carregar modelos e escolher pastas

O notebook monta o Drive e extrai automaticamente `JOE_MODELS.zip` para o armazenamento local da sessão Colab. Os modelos são carregados dos arquivos extraídos, não diretamente de dentro do ZIP. Também é possível informar um WAV ou `.safetensors` como voz padrão.

Estrutura esperada dentro do ZIP (com ou sem uma pasta externa `JOE_MODELS/`):
```text
JOE_MODELS.zip
├── pocket-tts/languages/portuguese/model.safetensors
├── pocket-tts-without-voice-cloning/languages/portuguese/tokenizer.json
└── huggingface-cache/
```

O arquivo deve estar em `Meu Drive/3d_models_ia/JOE_MODELS.zip`. O conteúdo será extraído para `/content/joe/models/` a cada nova sessão Colab. Se o ZIP for grande, aguarde a extração terminar antes de iniciar a API.


In [ ]:
import os
,from pathlib import Path
,import yaml
,import ipywidgets as widgets
,from google.colab import drive
,from IPython.display import display, clear_output
,
,drive.mount('/content/drive')
,
,DEFAULT_ZIP_PATH = Path('/content/drive/MyDrive/3d_models_ia/JOE_MODELS.zip')
,EXTRACT_BASE = Path('/content/joe/models')
,EXTRACTED_MODELS_ROOT = EXTRACT_BASE / 'JOE_MODELS'
,zip_widget = widgets.Text(
,    value=str(DEFAULT_ZIP_PATH),
,    description='Arquivo ZIP:',
,    layout=widgets.Layout(width='95%'),
,)
,DEFAULT_MODELS_ROOT = EXTRACTED_MODELS_ROOT
,model_folder_widget = widgets.Text(
,    value=str(DEFAULT_MODELS_ROOT / 'pocket-tts'),
,    description='Pasta Pocket TTS:',
,    layout=widgets.Layout(width='95%'),
,)
,tokenizer_widget = widgets.Text(
,    value=str(DEFAULT_MODELS_ROOT / 'pocket-tts-without-voice-cloning' / 'languages' / 'portuguese' / 'tokenizer.json'),
,    description='Tokenizer JSON:',
,    layout=widgets.Layout(width='95%'),
,)
,voice_widget = widgets.Text(
,    value='',
,    placeholder='Opcional: caminho de um .wav ou .safetensors no Drive',
,    description='Voz padrão:',
,    layout=widgets.Layout(width='95%'),
,)
,hf_cache_widget = widgets.Text(
,    value=str(DEFAULT_MODELS_ROOT / 'huggingface-cache'),
,    description='Cache HF/ASR:',
,    layout=widgets.Layout(width='95%'),
,)
,use_drive_tts_widget = widgets.Checkbox(
,    value=True,
,    description='Carregar os pesos do Pocket TTS a partir do Google Drive',
,)
,apply_paths_button = widgets.Button(
,    description='Aplicar pastas e modelos',
,    button_style='success',
,    icon='check',
,)
,paths_output = widgets.Output()
,
,def _extract_models_zip(zip_path: Path) -> Path:
,    if not zip_path.is_file():
,        raise FileNotFoundError(f'Arquivo ZIP não encontrado: {zip_path}')
,    EXTRACT_BASE.mkdir(parents=True, exist_ok=True)
,    # Extrai com validação dos caminhos para impedir entradas ZIP fora da pasta de destino.
,    with zipfile.ZipFile(zip_path) as archive:
,        for member in archive.infolist():
,            target = (EXTRACT_BASE / member.filename).resolve()
,            if not target.is_relative_to(EXTRACT_BASE.resolve()):
,                raise ValueError(f'Caminho inseguro dentro do ZIP: {member.filename}')
,        archive.extractall(EXTRACT_BASE)
,    # Aceita ZIP com os diretórios diretamente na raiz ou dentro de JOE_MODELS/.
,    candidates = [EXTRACT_BASE / 'JOE_MODELS', EXTRACT_BASE]
,    for candidate in candidates:
,        if (candidate / 'pocket-tts').is_dir():
,            return candidate
,    raise FileNotFoundError(
,        'ZIP extraído, mas não encontrei a pasta pocket-tts. '
,        'Confira se o ZIP contém pocket-tts/ e pocket-tts-without-voice-cloning/.'
,    )
,
,def apply_drive_model_paths(_=None):
,    with paths_output:
,        clear_output()
,        # Limpa opções antigas antes de aplicar os valores atuais.
,        os.environ.pop('JOE_TTS_CONFIG_PATH', None)
,        os.environ.pop('JOE_TTS_DEFAULT_VOICE', None)
,        try:
,            import zipfile
,            cache_dir = Path(hf_cache_widget.value.strip()).expanduser()
,            cache_dir.mkdir(parents=True, exist_ok=True)
,            os.environ['HF_HOME'] = str(cache_dir)
,            print(f'Cache Hugging Face/Parakeet: {cache_dir}')
,
,            if not use_drive_tts_widget.value:
,                print('Pocket TTS usará a configuração padrão e baixará os pesos necessários para o cache.')
,                return
,
,            zip_path = Path(zip_widget.value.strip()).expanduser()
,            extracted_root = _extract_models_zip(zip_path)
,            model_folder_widget.value = str(extracted_root / 'pocket-tts')
,            tokenizer_default = extracted_root / 'pocket-tts-without-voice-cloning' / 'languages' / 'portuguese' / 'tokenizer.json'
,            tokenizer_widget.value = str(tokenizer_default)
,            print(f'ZIP extraído; modelos disponíveis em: {extracted_root}')
,            model_root = Path(model_folder_widget.value.strip()).expanduser()
,            weights_file = model_root / 'languages' / 'portuguese' / 'model.safetensors'
,            tokenizer_file = Path(tokenizer_widget.value.strip()).expanduser()
,            missing = [str(path) for path in (weights_file, tokenizer_file) if not path.is_file()]
,            if missing:
,                raise FileNotFoundError(
,                    'Não encontrei estes arquivos. Confira os caminhos e o conteúdo do Drive:\n' + '\n'.join(missing)
,                )
,
,            import pocket_tts
,            package_config = Path(pocket_tts.__file__).resolve().parent / 'config' / 'portuguese.yaml'
,            if not package_config.is_file():
,                raise FileNotFoundError(f'Configuração base não encontrada no pacote: {package_config}')
,            config_data = yaml.safe_load(package_config.read_text(encoding='utf-8'))
,            config_data['weights_path'] = str(weights_file)
,            config_data['weights_path_without_voice_cloning'] = None
,            config_data['flow_lm']['lookup_table']['tokenizer_path'] = str(tokenizer_file)
,
,            voice_value = voice_widget.value.strip()
,            voice_file = None
,            if voice_value:
,                voice_file = Path(voice_value).expanduser()
,                if not voice_file.is_file():
,                    raise FileNotFoundError(f'Arquivo de voz não encontrado: {voice_file}')
,
,            runtime_dir = Path('/content/joe/runtime')
,            runtime_dir.mkdir(parents=True, exist_ok=True)
,            local_config = runtime_dir / 'portuguese_from_drive.yaml'
,            local_config.write_text(yaml.safe_dump(config_data, sort_keys=False, allow_unicode=True), encoding='utf-8')
,            os.environ['JOE_TTS_CONFIG_PATH'] = str(local_config)
,            if voice_file is not None:
,                os.environ['JOE_TTS_DEFAULT_VOICE'] = str(voice_file)
,                print(f'Voz padrão local: {voice_file}')
,            else:
,                print('Voz padrão: Rafael (português), carregada do Hugging Face quando necessário.')
,
,            print(f'Pesos Pocket TTS: {weights_file}')
,            print(f'Tokenizer: {tokenizer_file}')
,            print(f'Configuração local criada: {local_config}')
,            print('Configuração aplicada. A API usará estes caminhos quando receber language=portuguese.')
,        except Exception as exc:
,            print(f'Não foi possível aplicar as pastas: {exc}')
,            print('Ajuste os campos acima e clique novamente em Aplicar pastas e modelos.')
,
,apply_paths_button.on_click(apply_drive_model_paths)
,display(widgets.VBox([
,    use_drive_tts_widget,
,    zip_widget,
,    model_folder_widget,
,    tokenizer_widget,
,    voice_widget,
,    hf_cache_widget,
,    apply_paths_button,
,    paths_output,
,]))
,apply_drive_model_paths()


## Iniciar a API e o túnel

A API só aceita chamadas com o token gerado nesta sessão. O token não é salvo no repositório nem no disco pelo notebook. Copie-o para o Joe e evite compartilhar esta saída.


In [ ]:
import os
import re
import secrets
import shutil
import subprocess
import sys
import threading
import time
import queue
import requests
from IPython.display import display, Markdown

API_TOKEN = secrets.token_urlsafe(36)
os.environ['JOE_API_TOKEN'] = API_TOKEN

# Instala o binário oficial cloudflared para criar um Quick Tunnel temporário.
cloudflared_path = shutil.which('cloudflared')
if cloudflared_path is None:
    binary = '/usr/local/bin/cloudflared'
    subprocess.run(
        ['wget', '-q', '-O', binary,
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
        check=True,
    )
    os.chmod(binary, 0o755)
    cloudflared_path = binary

# Libera a porta 8000 antes de iniciar uma nova instância da API.
result = subprocess.run(
    ["bash", "-c", "fuser -k 8000/tcp"],
    capture_output=True,
    text=True
)

time.sleep(2)

print("Porta 8000 liberada.")

server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api_server:app',
     '--host', '127.0.0.1', '--port', '8000', '--no-access-log'],
    cwd='/content/joe/colab',
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Confirma que a API iniciou antes de publicar o túnel.
api_ready = False
for _ in range(60):
    if server_process.poll() is not None:
        raise RuntimeError('O servidor Joe API terminou inesperadamente.')
    try:
        response = requests.get(
            'http://127.0.0.1:8000/health',
            headers={'Authorization': f'Bearer {API_TOKEN}'},
            timeout=2,
        )
        if response.status_code == 200:
            api_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)
if not api_ready:
    raise RuntimeError('A API não respondeu. Confira a saída do processo uvicorn.')

tunnel_lines = queue.Queue()
tunnel_process = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def _read_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.put(line.rstrip())

threading.Thread(target=_read_tunnel_output, daemon=True).start()
API_URL = None
deadline = time.time() + 90
while time.time() < deadline and API_URL is None:
    if tunnel_process.poll() is not None:
        raise RuntimeError('O cloudflared terminou inesperadamente.')
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        API_URL = match.group(0)

if not API_URL:
    raise RuntimeError('Não foi possível obter a URL trycloudflare. Veja os logs do cloudflared.')

display(Markdown(
    '## Joe API pronta\n\n'
    f'**URL da API:** `{API_URL}`\n\n'
    f'**Token Bearer:** `{API_TOKEN}`\n\n'
    '**Copie os dois valores para a aba Google Colab / API do Joe.** '
    'Não compartilhe esta saída nem publique o token.'
))


## Encerrar a sessão

Execute esta célula quando terminar. Ela encerra o servidor e o túnel; ao iniciar novamente, será gerado outro token e outra URL.


In [ ]:
for process_name in ('tunnel_process', 'server_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
print('Joe API e túnel encerrados.')
